In [6]:
import random

import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [7]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [8]:
train_dataset = datasets.ImageFolder(root='data/train', transform=transform)
evaluation_dataset = datasets.ImageFolder(root='data/test/', transform = transform)

# Verify the loaded classes
print("Classes:", train_dataset.classes)
print("Training samples:", len(train_dataset))
print("Testing samples:", len(evaluation_dataset))


Classes: ['curly', 'straight']
Training samples: 800
Testing samples: 201


In [9]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [10]:
import torch.nn as nn


class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

In [12]:
model  = HairModel()

In [13]:
trainable_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters

20073473

In [7]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

NameError: name 'nn' is not defined